In [1]:
!nvidia-smi

[HAMI-core Msg(464:140096515639104:libvgpu.c:839)]: Initializing.....
Wed Aug 12 09:38:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:C2:00.0 Off |                    0 |
| N/A   58C    P0            110W /  300W |       0MiB /   8192MiB |      0%      Default |
|                                         |                        |  

In [ ]:
# Install or upgrade faster-whisper
#!pip install -U faster-whisper

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 143.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 113.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 160.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/4 [onnxruntime]  WARNING: The script onnxruntime_test is installed in '/home/jovyan/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [ctranslate2]  WARNING: The scripts ct2-fairseq-converter, ct2-marian-converter, ct2-openai-gpt2-converter, ct2-opennmt-py-converter, ct2-opennmt-tf-converter, ct2-opus-mt-converter and ct2-transformers-converter are installed in '/home/jovyan/.local/bin' which is not on P

In [ ]:
# Load Whisper large-v3 in FP16 on CUDA
from faster_whisper import WhisperModel
from pathlib import Path
import time
import os

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

print("Whisper large-v3 loaded")

cwd = Path.cwd()
print("cwd:", cwd)
print("cwd files:", os.listdir(cwd))
print("home:", Path.home())

search_name = "test.wav"
audio_path = None
for base in [cwd, Path.home(), Path("/home/jovyan")]:
    if not base.exists():
        continue
    for candidate in base.rglob(search_name):
        audio_path = candidate
        break
    if audio_path is not None:
        break

if audio_path is None:
    raise FileNotFoundError(
        f"{search_name} not found in cwd {cwd} or home {Path.home()}"
    )

print("Using audio file:", audio_path)

# Check GPU memory use after loading
!nvidia-smi

# Transcribe a short test WAV file with beam_size=1 and beam_size=5.
for beam_size in (1, 5):
    print(f"\nTranscribing with beam_size={beam_size}")
    start = time.perf_counter()
    segments, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=beam_size
    )
    segments = list(segments)
    elapsed = time.perf_counter() - start
    text = " ".join(segment.text.strip() for segment in segments)

    print("Transcript:")
    print(text)
    print()
    print(f"Audio duration: {info.duration:.3f}s")
    print(f"Transcription time: {elapsed:.3f}s")
    print(f"Realtime factor: {elapsed / info.duration:.3f}x")


Whisper large-v3 loaded
[HAMI-core Msg(544:140649443084096:libvgpu.c:839)]: Initializing.....
Wed Aug 12 09:44:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:C2:00.0 Off |                    0 |
| N/A   58C    P0            110W /  300W |    7742MiB /   8192MiB |      0%      Default |
|                                         |   

FileNotFoundError: [Errno 2] No such file or directory: 'test.wav'